In [2]:
import torch
import tensorflow as tf
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

from collections import Counter
from tqdm.auto import tqdm

import umap
import faiss



In [3]:
data = np.load("../data/dinov3_vmmrdb_full.npz", allow_pickle=True)

embeddings = data["embeddings"].astype("float32")
labels = data["labels"]

print(embeddings.shape)

(281752, 1024)


In [4]:
for start in range(0, len(embeddings), 5000):
    end = min(start + 5000, len(embeddings))
    
    norms = np.linalg.norm(embeddings[start:end], axis=1, keepdims=True)
    embeddings[start:end] /= norms

print("Normalization complete")

Normalization complete


In [5]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Index built")

Index built


In [6]:
import numpy as np
import os

print(os.path.getsize("../data/dinov3_vmmrdb_full.npz") / 1024**3, "GB")

data = np.load("../data/dinov3_vmmrdb_full.npz", mmap_mode="r", allow_pickle=True)

print(data.files)
print(data["embeddings"].shape)
print(data["embeddings"].dtype)

1.2700263783335686 GB
['embeddings', 'labels', 'paths']
(281752, 1024)
float32


In [7]:
data = np.load("../data/dinov3_vmmrdb_full.npz", allow_pickle=True)

print(data["embeddings"].shape)
print(data["embeddings"].dtype)
print(data["labels"].shape)

(281752, 1024)
float32
(281752,)


In [8]:
N = len(embeddings)
D = embeddings.shape[1]

normalized_path = "../data/vmmrdb_embeddings_normalized.npy"

normalized = np.lib.format.open_memmap(
    normalized_path,
    mode="w+",
    dtype="float32",
    shape=(N, D)
)

chunk_size = 5000

for start in tqdm(range(0, N, chunk_size)):
    end = min(start + chunk_size, N)

    x = embeddings[start:end]
    norms = np.linalg.norm(x, axis=1, keepdims=True)

    normalized[start:end] = x / norms

normalized.flush()

del normalized
del embeddings

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:00<00:00, 313.56it/s]


In [9]:
normalized = np.load(
    "../data/vmmrdb_embeddings_normalized.npy",
    mmap_mode="r"
)

print(normalized.shape)

(281752, 1024)


In [10]:
N = normalized.shape[0]

best_similarity = np.full(N, -np.inf, dtype=np.float32)
best_index = np.full(N, -1, dtype=np.int32)

query_chunk = 2000
database_chunk = 10000

for q_start in tqdm(range(0, N, query_chunk)):
    q_end = min(q_start + query_chunk, N)

    queries = normalized[q_start:q_end]

    for d_start in range(0, N, database_chunk):
        d_end = min(d_start + database_chunk, N)

        database = normalized[d_start:d_end]

        similarities = queries @ database.T

        # Don't let an image choose itself
        overlap_start = max(q_start, d_start)
        overlap_end = min(q_end, d_end)

        if overlap_start < overlap_end:
            rows = np.arange(
                overlap_start,
                overlap_end
            ) - q_start

            cols = np.arange(
                overlap_start,
                overlap_end
            ) - d_start

            similarities[rows, cols] = -np.inf

        local_indices = similarities.argmax(axis=1)
        local_values = similarities[
            np.arange(q_end - q_start),
            local_indices
        ]

        better = local_values > best_similarity[q_start:q_end]

        best_similarity[q_start:q_end][better] = local_values[better]
        best_index[q_start:q_end][better] = (
            d_start + local_indices[better]
        )

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 141/141 [01:51<00:00,  1.26it/s]


In [11]:
labels_arr = np.asarray(labels)

same_label = labels_arr == labels_arr[best_index]

print("Nearest-neighbor accuracy:", same_label.mean())

Nearest-neighbor accuracy: 0.42529245577671143


In [12]:
labels_arr = np.asarray(labels)

def make_model(label):
    return label.rsplit("_", 1)[0]

make_model_labels = np.array([
    make_model(label) for label in labels_arr
])

same_make_model = (
    make_model_labels == make_model_labels[best_index]
)

print("Make + model NN accuracy:", same_make_model.mean())

Make + model NN accuracy: 0.8920398080581504


In [13]:
def make_only(label):
    return label.split("_")[0]

make_labels = np.array([
    make_only(label) for label in labels_arr
])

same_make = make_labels == make_labels[best_index]

print("Make-only NN accuracy:", same_make.mean())

Make-only NN accuracy: 0.9777073454669355
